# Chapitre 12 - Exercices 12.6 à 12.11

* 12.6 : *optimal scoring* et problème aux valeurs propres ;
* 12.7 : *optimal scoring* pénalisé (PDA) ;
* 12.8 : les coefficients de l'*optimal scoring* sont proportionnels aux directions de la LDA ;
* 12.9 : LDA sur les valeurs ajustées d'une régression sur indicatrices ;
* 12.10 : LDA régularisée à noyau ;
* 12.11 : un modèle de mélange à centres partagés (MDA2) et son algorithme EM.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import eigh
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
from esl import datasets
from esl.ch12_svm import (LDA, FDA, MDA, class_indicator, second_diff_penalty, kmeans,
                          rbf_kernel, poly2_features, polynomial_kernel, simulate_mixture,
                          mixture_bayes_prob, plot_mixture_boundary, simulate_waveform)
set_style()

Notations communes : $Y$ est la matrice indicatrice $N\times K$ des classes,
$D_\pi = Y^TY/N = \mathrm{diag}(\hat\pi_1,\dots,\hat\pi_K)$ la matrice des proportions de
classes, $H$ la matrice $N\times M$ des $h_j(x_i)$. Nous utilisons les données *vowel*
(11 classes, 10 variables) pour les vérifications.

In [2]:
Xv, gv = datasets.load_vowel("train")
Yv, cls = class_indicator(gv)
N, K = Yv.shape
Dpi = Yv.T @ Yv / N
Hv = np.column_stack([np.ones(N), Xv])          # base : constante + variables
print("N = %d, K = %d, p = %d" % (N, K, Xv.shape[1]))

N = 528, K = 11, p = 10


## Exercice 12.6 : *Optimal scoring* et valeurs propres

**Énoncé (reformulé).** La régression de la FDA est un développement linéaire sur des
fonctions de base $h_m(x)$, $m = 1,\dots,M$.

(a) Montrer que le problème (12.52) s'écrit $\min_{\theta,\beta}\|Y\theta - H\beta\|^2$ (12.65),
où $\theta\in\mathbb R^K$.
(b) Interpréter les normalisations $\theta^TD_\pi\mathbf 1 = 0$ et $\theta^TD_\pi\theta = 1$.
(c) Montrer qu'après optimisation en $\beta$, on obtient $\max_\theta\theta^TY^TSY\theta$ (12.66)
sous ces contraintes, où $S$ est le projecteur associé à $H$.
(d) Si les $h_j$ contiennent la fonction constante, montrer que la plus grande valeur propre
de $S$ vaut 1.
(e) Avec une matrice $\Theta$ ($K\times K$) de scores normalisée par $\Theta^TD_\pi\Theta = I$,
montrer que la solution de (12.53) est donnée par l'ensemble des vecteurs propres ; le
premier est trivial et assure le centrage, les autres caractérisent la solution.